In [ ]:
# ============================================================
# TRUNCATION, ROUNDING AND QUANTIZATION ERROR
# ============================================================
#
# This notebook demonstrates the reduction of fixed-point word
# length using:
#
#     1. Rounding
#     2. Value truncation
#     3. Magnitude truncation
#
# It also displays:
#
#     - the quantizer characteristic Q(x),
#     - the quantization error e(x) = Q(x) - x,
#     - the quantization step q,
#     - the theoretical error bounds,
#     - the retained and discarded binary digits.
#
# DEFINITIONS
#
# Let K be the number of fractional bits retained after
# quantization. Then
#
#                       q = 2^(-K)
#
# is the quantization step.
#
# ROUNDING
#
# The quantized value is the nearest representable value.
#
#                       |e| <= q/2
#
# VALUE TRUNCATION
#
# The value is always quantized downward:
#
#                       Q(x) <= x
#
# and therefore
#
#                       -q < e <= 0.
#
# MAGNITUDE TRUNCATION
#
# The magnitude is reduced toward zero.
#
# For x > 0:
#
#                       -q < e <= 0
#
# For x < 0:
#
#                        0 <= e < q.
#
# IMPORTANT
#
# Overflow is deliberately excluded from this demonstration.
# The purpose is to study quantization independently.
#
# ============================================================


%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, IntSlider, RadioButtons, VBox, HBox, HTML, Layout, interactive_output
from IPython.display import display


# ------------------------------------------------------------
# Quantization functions
# ------------------------------------------------------------

def rounding_quantizer(x, q):

    x = np.asarray(x)

    return np.where(
        x >= 0,
        np.floor(x / q + 0.5) * q,
        np.ceil(x / q - 0.5) * q
    )


def value_truncation_quantizer(x, q):

    x = np.asarray(x)

    return np.floor(x / q) * q


def magnitude_truncation_quantizer(x, q):

    x = np.asarray(x)

    return np.sign(x) * np.floor(np.abs(x) / q) * q


def quantize(x, q, method):

    if method == 'Rounding':
        return rounding_quantizer(x, q)

    if method == 'Value truncation':
        return value_truncation_quantizer(x, q)

    return magnitude_truncation_quantizer(x, q)


# ------------------------------------------------------------
# Fixed-point binary helpers
# ------------------------------------------------------------

def fractional_binary(value, L):

    magnitude = abs(value)

    bits = []

    remainder = magnitude

    for _ in range(L):

        remainder *= 2.0

        bit = int(np.floor(remainder + 1e-12))

        bits.append(str(bit))

        remainder -= bit

    return ''.join(bits)


def render_binary_bits(bit_string, K):

    retained = []

    discarded = []

    for i, bit in enumerate(bit_string):

        if i < K:
            retained.append(
                f"<span class='q-bit q-retained'>{bit}</span>"
            )

        else:
            css = "q-rounding-bit" if i == K else "q-discarded"

            discarded.append(
                f"<span class='q-bit {css}'>{bit}</span>"
            )

    return ''.join(retained), ''.join(discarded)


# ------------------------------------------------------------
# Style
# ------------------------------------------------------------

style_html = HTML("""
<style>

.q-root {
    font-family: monospace;
    width: 900px;
    max-width: 900px;
}

.q-description {
    font-size: 13px;
    line-height: 1.45;
    padding: 9px 12px;
    border: 1px solid #bfc7d5;
    border-left: 6px solid #4a6fa5;
    background: #f7f9fc;
    border-radius: 8px;
    margin-bottom: 8px;
    box-sizing: border-box;
}

.q-box {
    border: 1px solid #c8d0dc;
    border-radius: 9px;
    padding: 9px 12px;
    box-sizing: border-box;
}

.q-title {
    font-size: 16px;
    font-weight: bold;
    color: #243447;
    margin-bottom: 6px;
}

.q-info {
    font-size: 14px;
    line-height: 1.55;
}

.q-label {
    display: inline-block;
    min-width: 185px;
    font-weight: bold;
}

.q-value {
    font-size: 16px;
    font-weight: bold;
}

.q-note {
    font-size: 12px;
    line-height: 1.35;
    color: #555555;
    margin-top: 5px;
}

.q-bit {
    display: inline-block;
    width: 27px;
    height: 29px;
    line-height: 29px;
    text-align: center;
    margin-right: 3px;
    border-radius: 5px;
    border: 1px solid #7f8c9a;
    font-size: 13px;
    font-weight: bold;
}

.q-retained {
    background: #dff3e4;
    color: #1d5d2d;
}

.q-rounding-bit {
    background: #fff3bf;
    color: #111111;
    border: 2px solid #d62828;
}

.q-discarded {
    background: #eeeeee;
    color: #777777;
}

.q-sign {
    display: inline-block;
    margin-right: 7px;
    font-size: 15px;
    font-weight: bold;
}

.q-binary-line {
    margin-top: 5px;
    margin-bottom: 5px;
    line-height: 36px;
}

.q-bound {
    font-weight: bold;
    color: #7a3d00;
}

</style>
""")


# ------------------------------------------------------------
# Title and description
# ------------------------------------------------------------

title_html = HTML("""
<div class="q-root">
    <div style="font-family:monospace;font-size:22px;font-weight:bold;margin-bottom:8px;">
        Truncation, Rounding and Quantization Error
    </div>
</div>
""")


description_html = HTML("""
<div class="q-root">

    <div class="q-description">

        Finite register length forces a high-precision value to be represented
        using fewer binary digits. This operation introduces a quantization
        error <b>e(x) = Q(x) - x</b>.<br>

        Use the controls to compare rounding, value truncation and magnitude
        truncation. The binary display shows the retained bits, the first
        discarded bit and the remaining discarded bits.<br>

        The two diagrams simultaneously show the quantizer characteristic
        <b>Q(x)</b> and its corresponding error characteristic <b>e(x)</b>.

    </div>

</div>
""")


# ------------------------------------------------------------
# Controls
# ------------------------------------------------------------

slider_layout = Layout(width='305px')

slider_style = {'description_width': '135px'}


x_slider = FloatSlider(
    value=0.710205078125,
    min=-0.95,
    max=0.95,
    step=2**(-12),
    description='Input x:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout,
    readout_format='.6f'
)


source_bits_slider = IntSlider(
    value=12,
    min=6,
    max=16,
    step=1,
    description='Source frac. bits:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)


retained_bits_slider = IntSlider(
    value=8,
    min=2,
    max=11,
    step=1,
    description='Retained bits K:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)


method_selector = RadioButtons(
    options=['Rounding', 'Value truncation', 'Magnitude truncation'],
    value='Rounding',
    description='Method:',
    style={'description_width': '80px'},
    layout=Layout(width='305px')
)


controls_box = VBox(
    [
        HTML("<div class='q-title'>Controls</div>"),
        x_slider,
        source_bits_slider,
        retained_bits_slider,
        method_selector
    ],
    layout=Layout(
        width='325px',
        border='1px solid #c8d0dc',
        padding='9px'
    )
)


summary_html = HTML()


summary_html.layout = Layout(
    width='560px',
    min_width='560px'
)


top_row = HBox(
    [
        summary_html,
        controls_box
    ],
    layout=Layout(
        width='900px',
        align_items='flex-start',
        justify_content='space-between'
    )
)


# ------------------------------------------------------------
# Update K when the source precision changes
# ------------------------------------------------------------

def update_source_bits(change):

    L = change['new']

    retained_bits_slider.max = L - 1

    if retained_bits_slider.value >= L:
        retained_bits_slider.value = L - 1

    x_slider.step = 2**(-L)


source_bits_slider.observe(
    update_source_bits,
    names='value'
)


# ------------------------------------------------------------
# Main interactive function
# ------------------------------------------------------------

def update_quantization(x, L, K, method):

    q = 2.0**(-K)

    x = round(x * 2**L) / 2**L

    Q = float(quantize(x, q, method))

    error = Q - x

    source_binary = fractional_binary(x, L)

    retained_bits, discarded_bits = render_binary_bits(
        source_binary,
        K
    )

    sign_symbol = '+' if x >= 0 else '-'

    rounding_bit = source_binary[K] if K < L else '---'


    if method == 'Rounding':

        theoretical_bound = f"-q/2 ≤ e ≤ q/2 = ±{q / 2:.8f}"

        interpretation = "The result is the nearest representable fixed-point value."


    elif method == 'Value truncation':

        theoretical_bound = f"-q < e ≤ 0, with q = {q:.8f}"

        interpretation = "Quantization is always downward on the real-number axis."


    else:

        if x >= 0:
            theoretical_bound = f"-q < e ≤ 0, with q = {q:.8f}"
        else:
            theoretical_bound = f"0 ≤ e < q, with q = {q:.8f}"

        interpretation = "The magnitude is reduced toward zero."


    summary_html.value = f"""
    <div class="q-box">

        <div class="q-title">
            Current Quantization
        </div>

        <div class="q-info">

            <span class="q-label">Original value</span>
            x = <span class="q-value">{x:.9f}</span>
            <br>

            <span class="q-label">Source precision</span>
            L = {L} fractional bits
            <br>

            <span class="q-label">Retained precision</span>
            K = {K} fractional bits
            <br>

            <span class="q-label">Quantization step</span>
            q = 2<sup>-{K}</sup> = {q:.9f}
            <br>

            <span class="q-label">Quantized value</span>
            Q(x) = <span class="q-value">{Q:.9f}</span>
            <br>

            <span class="q-label">Quantization error</span>
            e = Q(x) - x = <span class="q-value">{error:.9f}</span>

        </div>

        <div class="q-binary-line">

            <span class="q-sign">{sign_symbol}0.</span>

            {retained_bits}

            {discarded_bits}

        </div>

        <div class="q-note">
            Green bits are retained. The yellow/red bit is the first
            discarded bit and therefore the rounding decision bit.
            Gray bits are discarded.
        </div>

        <div class="q-info" style="margin-top:6px;">

            <span class="q-label">First discarded bit</span>
            {rounding_bit}
            <br>

            <span class="q-label">Theoretical error range</span>
            <span class="q-bound">{theoretical_bound}</span>

        </div>

        <div class="q-note">
            {interpretation}
        </div>

    </div>
    """


    # --------------------------------------------------------
    # Characteristic curves
    # --------------------------------------------------------

    xx = np.linspace(-0.95, 0.95, 4000)

    yy = quantize(xx, q, method)

    ee = yy - xx


    # --------------------------------------------------------
    # Quantizer characteristic Q(x)
    # --------------------------------------------------------

    fig1, ax1 = plt.subplots(figsize=(10.5, 3.3))

    ax1.plot(
        xx,
        xx,
        '--',
        linewidth=1.2,
        label='Ideal response  y = x'
    )

    ax1.plot(
        xx,
        yy,
        linewidth=1.8,
        label=f'{method}  Q(x)'
    )

    ax1.plot(
        x,
        Q,
        'o',
        markersize=8,
        label='Current operating point'
    )

    ax1.set_xlabel('Input x')

    ax1.set_ylabel('Quantized output Q(x)')

    ax1.set_title('Quantizer Input-Output Characteristic')

    ax1.grid(True, alpha=0.25)

    ax1.legend(
        loc='upper left',
        bbox_to_anchor=(1.02, 1.0)
    )

    plt.tight_layout()

    plt.show()

    plt.close(fig1)


    # --------------------------------------------------------
    # Error characteristic e(x)
    # --------------------------------------------------------

    fig2, ax2 = plt.subplots(figsize=(10.5, 2.8))

    ax2.plot(
        xx,
        ee,
        linewidth=1.8,
        label='e(x) = Q(x) - x'
    )

    ax2.axhline(
        0,
        linewidth=0.8
    )

    if method == 'Rounding':

        ax2.axhline(
            q / 2,
            linestyle='--',
            linewidth=1.0,
            label='+q/2'
        )

        ax2.axhline(
            -q / 2,
            linestyle='--',
            linewidth=1.0,
            label='-q/2'
        )

    elif method == 'Value truncation':

        ax2.axhline(
            -q,
            linestyle='--',
            linewidth=1.0,
            label='-q'
        )

    else:

        ax2.axhline(
            q,
            linestyle='--',
            linewidth=1.0,
            label='+q'
        )

        ax2.axhline(
            -q,
            linestyle='--',
            linewidth=1.0,
            label='-q'
        )

    ax2.plot(
        x,
        error,
        'o',
        markersize=8,
        label='Current error'
    )

    ax2.set_xlabel('Input x')

    ax2.set_ylabel('Error e(x)')

    ax2.set_title('Quantization Error Characteristic')

    ax2.grid(True, alpha=0.25)

    ax2.legend(
        loc='upper left',
        bbox_to_anchor=(1.02, 1.0)
    )

    plt.tight_layout()

    plt.show()

    plt.close(fig2)


interactive_plot = interactive_output(
    update_quantization,
    {
        'x': x_slider,
        'L': source_bits_slider,
        'K': retained_bits_slider,
        'method': method_selector
    }
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(style_html)

display(title_html)

display(description_html)

display(top_row)

display(interactive_plot)